# Challenge: Identifying Outliers Statistically

Two textbook rules for flagging unusual transaction amounts, compared on the same data:

* **Standard deviation (3σ):** flag amounts more than three standard deviations from the mean.
* **Interquartile range (IQR):** flag amounts beyond 1.5 × IQR from the first/third quartile.

Each rule is then combined with the 07:00–09:00 time window, and finally applied **per card holder** to see whether it improves the result.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from db_utils import get_engine, load_transactions

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

transactions = load_transactions(get_engine())
print(f"{len(transactions):,} transactions loaded")

3,500 transactions loaded


## Outlier detection helpers
Both functions return the lower and upper bound so that the cut-offs can be inspected, not just the flagged rows.

In [2]:
def std_bounds(amounts, k=3):
    """Bounds at mean +/- k standard deviations (population std. dev.)."""
    mu, sigma = amounts.mean(), amounts.std(ddof=0)
    return mu - k * sigma, mu + k * sigma


def iqr_bounds(amounts, k=1.5):
    """Tukey fences: Q1 - k*IQR and Q3 + k*IQR."""
    q1, q3 = np.percentile(amounts, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def flag(df, bounds):
    low, high = bounds
    return (df["amount"] < low) | (df["amount"] > high)

## Rule 1 — Standard deviation

In [3]:
bounds_std = std_bounds(transactions["amount"])
transactions["outlier_std"] = flag(transactions, bounds_std)

print(f"Bounds: {bounds_std[0]:.2f} to {bounds_std[1]:.2f}")
print(f"Flagged: {transactions['outlier_std'].sum()} of {len(transactions)}")

early_std = transactions[transactions["outlier_std"]].between_time("07:00", "09:00").sort_values("amount", ascending=False)
early_std[["holder_name", "amount", "category", "merchant_name"]]

Bounds: -565.25 to 646.83
Flagged: 77 of 3500


,holder_name,amount,category,merchant_name
date,,,,
2018-12-07 07:22:03,Robert Johnson,"1,894.00",bar,"Curry, Scott and Richardson"
2018-03-05 08:26:08,Crystal Clark,"1,617.00",bar,Mccarty-Thomas
2018-03-06 07:18:09,Nancy Contreras,"1,334.00",bar,Griffin-Woodard
2018-01-22 08:07:03,Crystal Clark,"1,131.00",restaurant,"Walker, Deleon and Wolf"
2018-09-26 08:48:40,Robert Johnson,"1,060.00",restaurant,"Jenkins, Peterson and Beck"
2018-09-06 08:28:55,Robert Johnson,"1,017.00",bar,"Jacobs, Torres and Walker"
2018-03-26 07:41:59,Laurie Gibbs,"1,009.00",coffee shop,Padilla-Clements
2018-12-14 08:51:41,Megan Price,748.00,pub,Johnson-Fuller


In [4]:
px.scatter(early_std.reset_index(), x="holder_name", y="amount", color="category",
           hover_data=["merchant_name", "date"],
           labels={"holder_name": "Card holder", "amount": "Amount ($)"},
           title="3σ outliers between 07:00 and 09:00")

## Rule 2 — Interquartile range

In [5]:
bounds_iqr = iqr_bounds(transactions["amount"])
transactions["outlier_iqr"] = flag(transactions, bounds_iqr)

q1, q3 = np.percentile(transactions["amount"], [25, 75])
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={q3 - q1:.2f}")
print(f"Bounds: {bounds_iqr[0]:.2f} to {bounds_iqr[1]:.2f}")
print(f"Flagged: {transactions['outlier_iqr'].sum()} of {len(transactions)}")

early_iqr = transactions[transactions["outlier_iqr"]].between_time("07:00", "09:00").sort_values("amount", ascending=False)
early_iqr[["holder_name", "amount", "category", "merchant_name"]]

Q1=3.74  Q3=14.65  IQR=10.91
Bounds: -12.63 to 31.02
Flagged: 110 of 3500


,holder_name,amount,category,merchant_name
date,,,,
2018-12-07 07:22:03,Robert Johnson,"1,894.00",bar,"Curry, Scott and Richardson"
2018-03-05 08:26:08,Crystal Clark,"1,617.00",bar,Mccarty-Thomas
2018-03-06 07:18:09,Nancy Contreras,"1,334.00",bar,Griffin-Woodard
2018-01-22 08:07:03,Crystal Clark,"1,131.00",restaurant,"Walker, Deleon and Wolf"
2018-09-26 08:48:40,Robert Johnson,"1,060.00",restaurant,"Jenkins, Peterson and Beck"
2018-09-06 08:28:55,Robert Johnson,"1,017.00",bar,"Jacobs, Torres and Walker"
2018-03-26 07:41:59,Laurie Gibbs,"1,009.00",coffee shop,Padilla-Clements
2018-12-14 08:51:41,Megan Price,748.00,pub,Johnson-Fuller
2018-04-01 07:17:21,Nancy Contreras,100.00,coffee shop,Padilla-Clements


In [6]:
px.scatter(early_iqr.reset_index(), x="holder_name", y="amount", color="category",
           hover_data=["merchant_name", "date"],
           labels={"holder_name": "Card holder", "amount": "Amount ($)"},
           title="IQR outliers between 07:00 and 09:00")

## Comparing the two rules

In [7]:
comparison = pd.DataFrame({
    "Upper cut-off ($)": [bounds_std[1], bounds_iqr[1]],
    "Flagged (all day)": [transactions["outlier_std"].sum(), transactions["outlier_iqr"].sum()],
    "Flagged 07:00–09:00": [len(early_std), len(early_iqr)],
    "Smallest flagged ($)": [transactions.loc[transactions["outlier_std"], "amount"].min(),
                             transactions.loc[transactions["outlier_iqr"], "amount"].min()],
}, index=["3σ", "IQR (1.5×)"])
comparison

,Upper cut-off ($),Flagged (all day),Flagged 07:00–09:00,Smallest flagged ($)
3σ,646.83,77,8,691.00
IQR (1.5×),31.02,110,9,57.00


### Observations

* **3σ flags 77 transactions; IQR flags 110.** Within 07:00–09:00 the counts are close (8 vs 9); the extra IQR flag is a $100 coffee-shop charge.
* The 3σ cut-off is high (≈ $647) because the few huge charges inflate the mean and standard deviation, so the rule is partly blinded by the very outliers it should catch. The IQR cut-off (≈ $31) is based on quartiles and is robust to them.
* That is why IQR catches **33 additional charges between $57 and $626** that 3σ misses. They are not false alarms from ordinary spending: no transaction falls between $23.13 and $57, so the data has a clean gap between normal charges (≤ $23.13) and unusual ones.
* Early-morning flagged charges are mainly **bars** (4 of the 8 3σ flags), followed by restaurants, a coffee shop and a pub.
* Both rules treat all card holders as one population. The next section tests whether a per-holder baseline changes the result.

## Improvement — IQR rule applied per card holder

In [8]:
def per_holder_flags(df, k=1.5):
    q = df.groupby("holder_id")["amount"].quantile([0.25, 0.75]).unstack()
    iqr = q[0.75] - q[0.25]
    upper = df["holder_id"].map(q[0.75] + k * iqr)
    return df["amount"] > upper

transactions["outlier_iqr_holder"] = per_holder_flags(transactions)

print(f"Flagged globally (IQR): {transactions['outlier_iqr'].sum()}")
print(f"Flagged per holder (IQR): {transactions['outlier_iqr_holder'].sum()}")
print(f"Flagged by both: {(transactions['outlier_iqr'] & transactions['outlier_iqr_holder']).sum()}")
print(f"Flagged only per holder: {(~transactions['outlier_iqr'] & transactions['outlier_iqr_holder']).sum()}")

transactions[transactions["outlier_iqr_holder"] & ~transactions["outlier_iqr"]].sort_values("amount", ascending=False)[
    ["holder_name", "amount", "category"]].head(10)

Flagged globally (IQR): 110
Flagged per holder (IQR): 110
Flagged by both: 110
Flagged only per holder: 0


,holder_name,amount,category
date,,,


### Conclusion

* A per-holder baseline flags **exactly the same 110 transactions** as the global IQR rule. Normal spending is homogeneous here — every holder's median is about $10 — so a global threshold is already sufficient for this dataset.
* On real portfolios, customers differ widely and a per-customer baseline matters: a $300 charge is normal for one customer and extreme for another.
* IQR is the better first-pass rule for skewed money data, because 3σ is distorted by the outliers themselves.
* Context such as **time of day** improves precision: the same charge is more suspicious at 03:00 than at lunchtime, which is why the 07:00–09:00 window above isolates the most striking cases.

The practical next step is to combine the three signals — robust statistic, per-customer baseline, time context — into a simple risk score.